# M1 — TinyStories and custom byte-level BPE
This notebook coordinates repository code. The BPE implementation stays in `src`.


In [2]:
REPO_URL = "https://github.com/shinbkb/mini-gpt.git"
COMMIT_SHA = "ce90663ee14017f501b0d73d9a25b963c4e9f668"
DATASET_REVISION = "f54c09fd23315a6f9c86f9dc80f725de7d8f9c64"  # Prefer a pinned Hugging Face commit
OFFLINE_SOURCE = None   # Kaggle Input path containing this repository
SOURCE_COMMIT = None    # Required for offline source
TRAIN_FILE = None       # Local JSONL/TXT train split if offline
VALIDATION_FILE = None  # Separate validation file


In [3]:
from pathlib import Path
import os, re, shutil, subprocess, sys
root = Path('/kaggle/working/mini-gpt')
if OFFLINE_SOURCE:
    assert SOURCE_COMMIT and re.fullmatch(r'[0-9a-fA-F]{40}', SOURCE_COMMIT)
    if not root.exists():
        shutil.copytree(OFFLINE_SOURCE, root, ignore=shutil.ignore_patterns('.venv', '__pycache__'))
else:
    assert re.fullmatch(r'[0-9a-fA-F]{40}', COMMIT_SHA)
    if not root.exists():
        subprocess.run(['git', 'clone', REPO_URL, str(root)], check=True)
    dirty = subprocess.check_output(['git','status','--porcelain'], cwd=root, text=True)
    if dirty.strip(): raise RuntimeError('Checkout has local changes; preserve them before checkout.')
    subprocess.run(['git','fetch','origin',COMMIT_SHA], cwd=root, check=True)
    subprocess.run(['git','checkout','--detach',COMMIT_SHA], cwd=root, check=True)
    actual = subprocess.check_output(['git','rev-parse','HEAD'], cwd=root, text=True).strip()
    assert actual.lower() == COMMIT_SHA.lower()
os.chdir(root)
if not OFFLINE_SOURCE:
    subprocess.run([sys.executable,'-m','pip','install','-r','requirements.txt'], check=True)
elif not (TRAIN_FILE and VALIDATION_FILE):
    raise ValueError('Offline mode needs separate train and validation files.')


Cloning into '/kaggle/working/mini-gpt'...
From https://github.com/shinbkb/mini-gpt
 * branch            ce90663ee14017f501b0d73d9a25b963c4e9f668 -> FETCH_HEAD
HEAD is now at ce90663 m1 tokenizer


In [4]:
subprocess.run([sys.executable,'-m','pytest','-q'], check=True)


.....................                                                    [100%]
21 passed in 9.26s


CompletedProcess(args=['/usr/bin/python3', '-m', 'pytest', '-q'], returncode=0)

## Prepare tokenizer and bounded token caches
Default sample: first 20,000 train stories and 1,000 validation stories. The tokenizer is fitted on train only.
For reproducibility, set a dataset revision. Offline mode reads two distinct local files.


In [5]:
command = [sys.executable,'-m','src.data.prepare','--train-limit','20000',
           '--validation-limit','1000','--vocab-size','8000','--block-size','512',
           '--require-full-vocab']
if TRAIN_FILE and VALIDATION_FILE:
    command += ['--train-file',TRAIN_FILE,'--validation-file',VALIDATION_FILE]
else:
    if not DATASET_REVISION:
        raise ValueError('Set DATASET_REVISION to pin the online dataset version.')
    command += ['--revision',DATASET_REVISION]
subprocess.run(command, check=True)


{
  "dataset": "roneneldan/TinyStories",
  "dataset_revision": "f54c09fd23315a6f9c86f9dc80f725de7d8f9c64",
  "sample_policy": "first N records, no shuffle",
  "tokenizer_sha256": "dbabcd62e9592a44fb3b68980324e2c6a6ceea062060cc048a7fb0d1333023d0",
  "vocab_size": 8000,
  "target_vocab_size": 8000,
  "tokenizer_train_seconds": 3.907935385999963,
  "block_size": 512,
  "splits": {
    "train": {
      "stories": 20000,
      "tokens": 7731976,
      "windows": 15101,
      "encode_seconds": 45.75675592300013,
      "tokens_per_second": 168979.98654037967,
      "sample_sha256": "4109ac8f2b5831a8dff4e271d58ee29dbe5a8982de18c5fa8bd9d0ab3e924e23"
    },
    "validation": {
      "stories": 1000,
      "tokens": 339797,
      "windows": 663,
      "encode_seconds": 2.2344938940000247,
      "tokens_per_second": 152068.88723769152,
      "sample_sha256": "21a3cc78b2729b6eb1c6216f982b90811a6af153ba9f700684983231deb11fbb"
    }
  }
}


CompletedProcess(args=['/usr/bin/python3', '-m', 'src.data.prepare', '--train-limit', '20000', '--validation-limit', '1000', '--vocab-size', '8000', '--block-size', '512', '--require-full-vocab', '--revision', 'f54c09fd23315a6f9c86f9dc80f725de7d8f9c64'], returncode=0)

In [7]:
from src.tokenizer.bpe import ByteLevelBPE
tokenizer = ByteLevelBPE.load('artifacts/tokenizer.json')
assert tokenizer.vocab_size == 8000
sample = 'Tiếng Việt, emoji 🙂 and punctuation!\n'
assert tokenizer.decode(tokenizer.encode(sample)) == sample
print('Tokenizer round-trip OK; save/download /kaggle/working/mini-gpt/artifacts from Notebook Output.')


Tokenizer round-trip OK; save/download /kaggle/working/mini-gpt/artifacts from Notebook Output.


In [8]:
import shutil
from IPython.display import FileLink, display

zip_path = shutil.make_archive(
    "/kaggle/working/mini-gpt-artifacts",
    "zip",
    "/kaggle/working/mini-gpt/artifacts",
)
display(FileLink(zip_path))

/kaggle/working/mini-gpt-artifacts.zip